# Playwright: Web Scraping
Web data can be extracted through three main approaches:
- HTML parsing: Retrieve an HTML response and parse its elements. This works when the required data is present in the response.
- API requests: Call an official or internal HTTP endpoint and parse its structured response, commonly JSON.
- Browser automation: Run a browser when the page requires JavaScript execution or user interaction.

When a suitable API is available, it is usually simpler and more reliable than parsing presentation-oriented HTML.

In [0]:
import re
import time
from urllib.parse import urljoin
import requests

import bs4
import pandas as pd

from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains

## 1. Request and parsing
There are two basic ways to scrape data from websites:
- Server-side: You retrieve structured or unstructured HTML - the content displayed in a browser. It is human-readable but not always machine-readable. Most of the time, the data is present unless the site owner has actively hidden or protected it. You need a basic understanding of HTML to use this approach effectively.
- Client-side/API-based: You use REST APIs or JSON endpoints. When a site provides these endpoints, they often return well-structured data that is easier to extract and process. When possible, this approach tends to be simpler.

### 1.1. Requests
Instead of using a browser, you can use the [`requests`] library to retrieve a page's content. You can check the response status, such as `200 OK`, and then parse the content. This approach works well when the required data is delivered directly as HTML or JSON without complex front-end processing.

[`requests`]: https://github.com/psf/requests

In [0]:
import requests

In [0]:
url = 'https://books.toscrape.com/index.html'
response = requests.get(url)
response

In [0]:
response.text[:1000]

In [0]:
response.headers

### 1.2. Crawling APIs
Not all URLs return HTML. Some (REST API endpoints) return JSON directly, like https://api.github.com/repos/dmlc/xgboost. You can load that into Python structures via `response.json()`. However, real-world APIs often involve extra stuff: headers, authentication, maybe pagination, or request payloads.

In [0]:
import pandas as pd
import requests

#### Documented APIs
Many organizations officially support REST APIs for data access, such as [GitHub], [Facebook], [Twitter], [Reddit] and [Clash Royale]. To start using APIs provided this way, developers usually need to register an account and generate an API key, but some of them don't require any authentication. Either way, the instructions for requesting data can be found in their documentation sites.

[GitHub]: https://docs.github.com/en/rest
[Facebook]: https://developers.facebook.com/docs/groups-api/reference
[Twitter]: https://developer.twitter.com/en/docs/twitter-api
[Reddit]: https://www.reddit.com/dev/api/
[Clash Royale]: https://developer.clashroyale.com/

Now let's walk through an example by requesting GitHub's [list-repository-languages] endpoint to show the size (in bytes) of code written in each language.
- The main component of a request is the URL, which follows a predefined syntax. In this case, the URL has two placeholders for `OWNER` and `REPO`. They can be handled nicely using Python formatted strings. We can use this URL to access data from any public repository.
- For private repositories, we must provide an authentication key with appropriate permissions. This extra info goes in the request headers, you can think of them as metadata of the API call.
- We might notice that there are different requesting methods available such as GET, POST, PUT and DELETE that serve different purposes. As we only want to collect data, we only need to care about GET, and sometimes, POST.

[list-repository-languages]: https://docs.github.com/en/rest/repos/repos#list-repository-languages

In [0]:
OWNER = 'dmlc'
REPO = 'xgboost'

url = f'https://api.github.com/repos/{OWNER}/{REPO}/languages'
response = requests.get(url)
response.json()

In [0]:
OWNER = 'hungpq7'
REPO = 'courses'

url = f'https://api.github.com/repos/{OWNER}/{REPO}/languages'
headers = {
    'Accept': 'application/vnd.github+json',
    'Authorization': '',
    'X-GitHub-Api-Version': '2022-11-28',
}
response = requests.get(url, headers=headers)
response.json()

:::{note}
We can make API calls with command line too, using the [cURL](https://en.wikipedia.org/wiki/CURL) command.
:::

In [0]:
%%bash
curl https://api.github.com/repos/hungpq7/courses/languages \
    -H "Accept: application/vnd.github+json" \
    -H "Authorization: " \
    -H "X-GitHub-Api-Version: 2022-11-28"

#### Hidden APIs
The increasing popularity of JavaScript-based frameworks such as React, Angular and Vue encourages websites to be rendered *client-side*. This means, more websites use REST APIs to send and receive data to fill HTML templates, then render the page on user's computers. Of course, such APIs are not documented, so using them takes a bit of detective work including finding them and understanding their structures. In this section, we are going to inspect a page's network activities to locate scrapable APIs. 

::::{admonition} Case
:class: tip
We will be crawling all articles on the homepage of https://techcrunch.com/ using the following steps:
- Go to the target page and open the browser's developer tool. The shortcut in Google Chrome is <kbd>F12</kbd> or <kbd>Ctrl</kbd> <kbd>Shift</kbd> <kbd>I</kbd>. However, the tool will not record activities before it was opened, so we need to press <kbd>Ctrl</kbd> <kbd>R</kbd> to reload the target page.
- Navigate to the Network tab to show all requests the page has made and filter Fetch/XHR requests. This filter leaves only requests that fetch JSON data, separating them from other types of response that we don't need such as image, media and CSS. These buttons are yellow circled in the image below.

:::{image} ../image/chap_01/rest_api_response.png
:height: 300px
:align: center
:::
<br>

- At this point, one of the displaying requests returns the data we are looking for. We will need to explore a bit to determine that API, start with names. In the example of TechCrunch, the API "magazine" sounds promising. Indeed, when we click this API and preview its response, we see a list of items storing articles in the website.
- Now we have found the API, let's learn how to use it. Switching to the Header tab reveals to us the URL and the request method of this API. Other APIs may require headers as well as payload, but this one is not the case.

:::{image} ../image/chap_01/rest_api_url.png
:height: 300px
:align: center
:::
::::

In [0]:
url = 'https://techcrunch.com/wp-json/tc/v1/magazine?page=1&_embed=true&cachePrevention=0'
response = requests.get(url)
response

In [0]:
data = []
for item in response.json():
    sample = {
        'id': item['id'],
        'category': item['primary_category']['slug'],
        'author': item['parselyMeta']['parsely-author'][0],
        'title': item['parselyMeta']['parsely-title'],
    }
    data.append(sample)

pd.DataFrame.from_dict(data).head()

::::{admonition} Case
:class: tip
Sometimes, websites block connections from non-browser clients. For example, when inspecting the website https://tiki.vn/nha-sach-tiki/c8322, I found an API named "listing" which contains all products shown in the page. With the naked URL, we can access its response using Chrome but will get blocked using Requests. This can be easily bypassed by we overwriting the *user agent* header as follows.

:::{image} ../image/chap_01/rest_api_payload.png
:height: 300px
:align: center
:::

Now, if switch to the Payload tab, we can observe that the parameters here match exactly the components in the URL. With this insight, we can rewrite the request with URL and payload separatedly, which is far more readable.
::::

In [0]:
requests.utils.default_headers()

In [0]:
url = 'https://tiki.vn/api/personalish/v1/blocks/listings?limit=40&category=8322&page=1&urlKey=nha-sach-tiki'
headers = {'user-agent': 'Mozilla/5.0 Chrome/108.0.0.0 Safari/537.36'}
response = requests.get(url, headers=headers)
response

In [0]:
url = 'https://tiki.vn/api/personalish/v1/blocks/listings'
headers = {'user-agent': 'Mozilla/5.0 Chrome/108.0.0.0 Safari/537.36'}
payload = {
    'limit': 40,
    'category': 8322,
    'page': 1,
    'urlKey': 'nha-sach-tiki',
}
response = requests.get(url, headers=headers, params=payload)
response

In [0]:
for product in response.json()['data'][:5]:
    name = product['name']
    print(name)

## 2. HTML parser

### 2.1. HTML concepts
[HTML] is a language for creating web pages. The easiest way to think about HTML, is a language with the same purpose as Markdown, with less readability but more expressivity. A HTML document is built from *elements*, organized in a hierarchical structure. For example, here are the components of an element:
- The *tags* (element names), some are singleton (like `<br>`) while others come in pairs (like `<span>` `<\span>`)
- The text between two tags `computer` is the *content* of that element.
- An element can have a number of *attributes* such as `class` `style`.

[HTML]: https://en.wikipedia.org/wiki/HTML

In [0]:
%%html
<span class='breadcrumb content' style='color:indianred'>computer</span>

:::{note}
Some global attributes occur everywhere in HTML documents. Keeping an eye on them will help you a lot in scraping data:
- The attribute `id` is the identifier of an element, must be unique across the document. Useful in locating a specific element.
- The attribute `class` makes reference to custom CSS styles. Useful in matching a list of items with the same style. An element can have multiple classes, for example *breadcrumb* and *content*.
:::

### 2.2. Parsing
We use [Beautiful Soup], a great tool for navigating and extracting data from HTML. Steps usually are:
- Feeding the HTML string to it, you get a *soup* object
- Use tree navigation or searching to locate elements
- Extract text, attributes, links, etc.

[Beautiful Soup]: https://www.crummy.com/software/BeautifulSoup/bs4/doc/

In [0]:
import bs4

In [0]:
html = """
<html>
    <head>
        <title>The Dormouse's story</title>
    </head>
    <body>
        <p class="story">Once upon a time there were three little sisters; and their names were
            <a href="https://example.com/elsie" class="sister" id="link1">Elsie</a>,
            <a href="http://example.com/lacie" class="sister" id="link2">Lacie</a> and
            <a href="http://example.com/tillie" class="sister" id="link3">Tillie</a>;
            and they lived at the bottom of a well.
        </p>
        <time class="story">2000-01-01 06:00:00</time>
"""

In [0]:
soup = bs4.BeautifulSoup(html, 'html.parser')

#### Tree navigation
We can easily navigate a HTML document as BS has registered child tags and attributes to the *soup*. This syntax is simple, but the downside is that it cannot handle multiple children (only the first child is returned).

In [0]:
tag = soup.body.p.a

In [0]:
tag.text

In [0]:
tag['href']

#### Element searching
BS supports a more reliable way for finding exactly the element we want, via the [`find_all()`] method. This function searches for HTML tags and attributes:
- The [first argument] is the tag you want to find. It can be a string or list of strings, a regex pattern or a function.
- Other arguments are named after HTML attributes. But if some of them make conflicts to Python built-in names such as `id`, `class`, and `custom-attribute`, we can pass them as a dictionary to the `attrs` argument.

The ideal case in searching is when you know the ID of an element, thanks to its uniqueness. In this case, we can safely use the `find()` method instead, which returns only the first result. Otherwise, combinations of tags and attributes will help you in finding the element you want very quickly.

[`find_all()`]: https://www.crummy.com/software/BeautifulSoup/bs4/doc/#find-all
[first argument]: https://www.crummy.com/software/BeautifulSoup/bs4/doc/#kinds-of-filters

In [0]:
import re
import bs4

In [0]:
soup.find(id='link2')

In [0]:
soup.find_all(re.compile('^t'))

In [0]:
soup.find_all('a', class_='sister')

In [0]:
attrs = {
    'class': 'sister',
    'href': re.compile('https\S+')
}

soup.find_all('a', attrs=attrs)

:::{admonition} Case
:class: tip
Let's use Beautiful Soup to crawl all fiction books in the page https://books.toscrape.com/catalogue/category/books/fiction_10/index.html. This website does not use REST APIs, so the only way is parsing its HTML souce. Our crawler contains two phases, (1) gathering book URLs and (2) actually crawling book information.

*Phase 1*
- First, examine the URL structure to find that we can replace "index" with "page-n" to access pages. The index of page will be set incrementally, as we will get an 404 error message when it exceeds the maximum number.
- In the first page, use the *inspect* tool of Chrome to find book containers (each contains image, title, ratings and price). We observe that each container has 4 classes `col-xs-6` `col-sm-4` `col-md-3` `col-lg-3`.
You can re-check this information by searching and counting the number of elements that use all 4 classes. There are 20 of them, which matches the number of books the page shows.
- All the information in the containers also appear in book pages. So, the only data we get here is book URLs. Note that URLs here are relative, we can easily convert in into full path using the function [`urljoin()`]

*Phase 2*
- Access each URL collected in the first phase. Locate the content container.
- Extract the important fields and add them to a Pandas dataframe. There is no new technique in this phase.
:::

[`urljoin()`]: https://docs.python.org/3/library/urllib.parse.html#urllib.parse.urljoin

In [0]:
import re
import requests
import bs4
import pandas as pd
import time
from urllib.parse import urljoin

In [0]:
def crawl_book_url(url_base) -> list:
    list_url_book = []
    n_page = 1
    while True:
        # access crawl page and check if it loads successfully (status 200)
        url_page = url_base.replace('index', f'page-{n_page}')
        response = requests.get(url_page)
        if response.status_code != 200:
            break
        
        # create soup
        html = response.text
        soup = bs4.BeautifulSoup(html, 'html.parser')
        
        # get book containers
        list_container = soup.find_all('li', class_='col-xs-6 col-sm-4 col-md-3 col-lg-3')
        
        # get book urls
        for container in list_container:
            href = container.article.h3.a['href']
            url_book = urljoin(url_base, href)
            list_url_book.append(url_book)
        
        # advance to the next page
        n_page += 1
    
    return list_url_book

In [0]:
def crawl_book_info(list_url_book:list):
    data = []
    for url_book in list_url_book:
        response = requests.get(url_book)
        html = response.content
        soup = bs4.BeautifulSoup(html, 'html.parser')
        
        container = soup.find('article', class_='product_page')
        title = container.find('div', class_='product_main').h1.text
        description = container.find('p', class_=False).text
        
        table = container.find('table', class_='table-striped').prettify()
        table = pd.read_html(table)
        table = pd.concat(table)
        table = table.set_index(0)[1]
        
        upc = table['UPC']
        price = table['Price (excl. tax)']
        price = float(re.findall('\d+\.\d+', price)[0])
        tax = table['Tax']
        tax = float(re.findall('\d+\.\d+', tax)[0])
        
        sample = {
            'upc': upc,
            'title': title,
            'price': price,
            'tax': tax,
        }
        data.append(sample)
        
    return pd.DataFrame(data)

In [0]:
url_base = 'https://books.toscrape.com/catalogue/category/books/fiction_10/index.html'
list_url_book = crawl_book_url(url_base)
len(list_url_book)

In [0]:
df = crawl_book_info(list_url_book[:5])
df

## 3. Web driver
Sometimes, websites require user-interaction: scrolling, clicking, form inputs, etc. Beautiful Soup + Requests don’t run JS or simulate user actions. That’s when [Selenium], a *web driver*, becomes useful.

[Selenium]: https://github.com/SeleniumHQ/selenium

### 3.1. Driver initialization
- Use a browser driver ([Chrome Driver] is common).
- To avoid manually downloading binaries, you can use [Webdriver Manager].
- [Configure] the driver: headless mode, maximized, [page load strategy], etc.
- Quit the driver to free up memory.

[Chrome Driver]: https://chromedriver.chromium.org/home
[Webdriver Manager]: https://github.com/SergeyPirogov/webdriver_manager
[Configure]: https://www.selenium.dev/documentation/webdriver/drivers/options/
[page load strategy]: https://www.selenium.dev/documentation/webdriver/drivers/options/#pageloadstrategy

In [0]:
import time
from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

In [0]:
service = Service(ChromeDriverManager().install())
options = Options()
# options.add_argument('--headless')
# options.add_argument('--window-size=1920,1080')
options.add_argument('start-maximized')

driver = webdriver.Chrome(service=service, options=options)

time.sleep(5)
driver.quit()

### 3.2. Element locating
Selenium can find elements using various strategies, using the [`find_elements()`] method with a [`By`] locator.
- Basic methods: tag names, classes, attributes
- XPath: powerful, flexible; absolute vs. relative paths
- CSS selectors: usually cleaner and faster; many use these by copying from browser DevTools

[`find_elements()`]: https://www.selenium.dev/documentation/webdriver/elements/finders
[`By`]: https://selenium-python.readthedocs.io/locating-elements.html

:::{note}
You can simply use them by right-click an element and copy its XPath or CSS selector, but they're worth being learned carefully.
:::

#### XPath
The general syntax of XPath is `/element/element/...`, starts from root. You can leave the first element blank `//element/element/...` to turn the absolute path into relative. The basic syntaxes for searching elements are `tag[@attr='value']` and `tag[ordinal]`. For example:
- `*[@id='promotion']` matches any tag that has `id='promotion'`
- `div[@class='breadcrumb']` matches elements that have `<div class='breadcrumb'>`
- `span[7]` matches any `<span>` element that is the seventh child of its parent.

#### CSS selector
The general syntax of CSS selector is `element > element > ...`, being relative by nature. The basic syntaxes for searching elements are `tag.class` `tag#id` `tag[attr=value]` `tag:func(args)`. For example:
- `#promotion` matches any tag that has `id='promotion'`
- `div.breadcrumb` matches elements that have `<div class='breadcrumb'>`
- `span[role=alert]` matches elements that have `<span role='alert'>`
- `span:nth-child(7)` matches any `<span>` element that is the seventh child of its parent.

#### Implementation
In this section, we attempt to crawl first 20 books in the fiction category. We try traditional way first, using tags and classes, only to know that Selenium isn't very good at supporting this style. Next, we try to use XPath and CSS selector by copying those of a single book from Chrome and then tweaking them to match all 20.

In [0]:
import time
from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

In [0]:
service = Service(ChromeDriverManager().install())
options = Options()
options.add_argument('start-maximized')
driver = webdriver.Chrome(service=service, options=options)

In [0]:
url = 'https://books.toscrape.com/catalogue/category/books/fiction_10/index.html'
driver.get(url)

In [0]:
list_book = driver.find_elements(By.CLASS_NAME, 'col-xs-6')
len(list_book)

In [0]:
xpath = '//*[@id="default"]/div/div/div/div/section/div[2]/ol/li'
list_book = driver.find_elements(By.XPATH, xpath)
len(list_book)

In [0]:
selector = '#default > div > div > div > div > section > div:nth-child(2) > ol > li'
list_book = driver.find_elements(By.CSS_SELECTOR, selector)
len(list_book)

In [0]:
driver.quit()

### 3.3. Actions
In this section we use Selenium to perform basic [actions] on a website: clicking, typing keys and hovering.

[actions]: https://selenium-python.readthedocs.io/navigating.html

In [0]:
import time
from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains

In [0]:
service = Service(ChromeDriverManager().install())
options = Options()
options.add_argument('start-maximized')
driver = webdriver.Chrome(service=service, options=options)

In [0]:
url = 'https://www.tensorflow.org/'
driver.get(url)

#### Send keys

In [0]:
path = '/html/body/section/devsite-header/div/div[1]/div/div/div[2]/devsite-search/form/div[1]/div/input'
element = driver.find_element(By.XPATH, path)

In [0]:
element.send_keys('lstm')

In [0]:
element.send_keys(Keys.CONTROL, 'A')

In [0]:
element.send_keys(Keys.CONTROL, 'C')

In [0]:
element.send_keys(Keys.DOWN)

In [0]:
element.clear()

#### Hover

In [0]:
chains = ActionChains(driver)

In [0]:
path = '/html/body/section/devsite-header/div/div[1]/div/div/div[2]/div[1]/devsite-tabs/nav/tab[6]/a[1]'
element = driver.find_element(By.XPATH, path)
chains.move_to_element(element).perform()

In [0]:
path = '/html/body/section/devsite-header/div/div[1]/div/div/div[2]/div[1]/devsite-tabs/nav/tab[5]/a[1]'
element = driver.find_element(By.XPATH, path)
chains.move_to_element(element).perform()

In [0]:
element.click()

## Resources
- gregreda.com - [Web Scraping 201: finding the API](http://www.gregreda.com/2015/02/15/web-scraping-finding-the-api/)
- jovian.ai - [Introduction to Web Scraping and REST APIs](https://jovian.ai/aakashns/python-web-scraping-and-rest-api)
- blog.devgenius.io - [Scrape Data without Selenium by Exposing Hidden APIs](https://blog.devgenius.io/scrape-data-without-selenium-by-exposing-hidden-apis-946b23850d47)
- medium.com - [Web Crawling Made Easy with Scrapy and REST API](https://medium.com/@geneng/web-crawling-made-easy-with-scrapy-and-rest-api-ed993e84abd3)
- w3schools.com - [XPath syntax](https://www.w3schools.com/xml/xpath_syntax.asp)
- w3schools.com - [CSS selector reference](https://www.w3schools.com/cssref/css_selectors.php)